## Prompts for Analyze Topics Feature

### Setting up DSPy

In [ ]:
# Import libraries
import dspy
import json
import tiktoken

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [4]:
# # create gpt-4 model
# gpt4 = dspy.OpenAI(model='gpt-4-0125-preview', max_tokens=4000, api_key=gpt4_open_ai_api_key)  
# dspy.configure(lm=gpt4)

# gpt4("which openai model are you? Are you gpt4?")

In [ ]:
# Set up the LM (https://dspy-docs.vercel.app/api/language_model_clients/OpenAI)
gpt3_turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=4000, api_key=open_ai_api_key)  
dspy.configure(lm=gpt3_turbo)

### Declaring some global constants

In [4]:
# "relevance": number from 1 to 10,
# "question_diversity": number from 1 to 10,

constants = {
    "question_json_format": """{
        "cell_type": "question",
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }""",
    "analyze_topic_question_json_format": """{
        "section_id": number,
        "cell_type": "question",
        "response_format": "open" or "closed",
        "time_estimate": number of minutes,
        "rationale": string,
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }""",
    "analyze_topic_section_json_format": """{
        "id": section number starting from 0,
        "title": string,
    }""",
    "json_formatting_message": """Do not include any markdown formatting like triple quotes or backticks around your JSON output. If your output includes any markdown formatting, remove it before considering your output complete."""
}

### Helper functions

In [5]:
def parse_json_str(json_str):
    return json.loads(json_str)

def fill_in_constants(input_str):
    for key in constants:
        input_str = input_str.replace("{"+key+"}", constants[key])
    return input_str

In [6]:
# post-processing function to remove everything before the first square bracket and after the last square bracket
def post_process(output_str):
    output_str = output_str[output_str.find("["):]
    output_str = output_str[:output_str.rfind("]") + 1]
    return output_str

### Signature for creating new questions for a topic

In [8]:
sig_description = """
You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided.
2. Please read through the existing questions for the inputted topic.
3. Identify any gaps that are related to the topic that the existing questions do not cover.
4. For each gap, generate two to three questions that would help the user elicit useful information from their constituents.
5. Generate time estimates in minutes for each question, which should populate the time_estimate field in the JSON output.
6. Please review the additional questions and compose a detailed explanation for how this question differs from the existing questions. Please keep your response between 20 and 50 words. Please start your response with "This question" followed by your rationale. For example: "This question is being asked in order to..." These rationales should populate the rationale field in the JSON output.
7. Determine which sections from the inputted sections each question should be added to and provide the section_id in the JSON output.
"""
# 7. Return the gaps in the existing questions and the additional questions you have generated in the output fields.

# sig_description = """
# You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

# Please think step-by-step and follow these instructions carefully.
# 1. Please read and re-read the context that the user has provided.
# 2. Please read through the existing questions for the inputted topic.
# 3. Identify any gaps that are explicitly related to the topic that the existing questions do not cover.
# """

# sig_description = """
# You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

# Please think step-by-step and follow these instructions carefully.
# 1. Please read and re-read the context that the user has provided.
# 2. Please read through the existing questions for the inputted topic.
# 3. Generate three to five additional questions for the inputted topic.
# 4. Each question should have two scores. The first score, called "relevance", is how relevant the question is to the context. The second, "question_diversity", is a score that measures how different the question is from the existing set of questions, with lower scores being more similar to the existing questions and higher scores being more different. The scores should be from 1 to 10.
# 5. Generate time estimates in minutes for each question, which should populate the time_estimate field in the JSON output.
# 6. Please review the questions provided and compose a detailed explanation for the gap that the question is filling. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response with “This question” followed by your rationale. For example: “This question is being asked in order to... These rationales should populate the description field in the JSON output.
# """

input_descriptions = """{
    "context": "The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question.",
    "topic": "The topic for which the user wants to generate questions.",
    "existing_questions": "The existing questions for the inputted topic. The questions are organized as a list of JSONs. Each JSON follows the following structure: {question_json_format}",
    "sections": "The current sections in the survey or interview guide. The sections are organized as a list of JSONs. Each JSON follows the following structure: {analyze_topic_section_json_format}"
}
"""

output_descriptions = """{
    "additional_questions": "Additional questions for a topic. The output should be a list of JSONs surrounded by square brackets, where each element has the following structure: {analyze_topic_question_json_format}. Do not number the items in the list.",
    "gaps": "A list of gaps in the existing questions. The list should be a string where each gap is separated by a semicolon. Do not number the items in the list."
}"""

# output_descriptions = """{
#     "gaps": "A list of gaps in the existing questions. The list should be a string where each gap is separated by a semicolon. Do not number the items in the list."
# }"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class AddQuestionsToTopic(dspy.Signature):

    context = dspy.InputField(desc=fill_in_constants(input_descriptions_json["context"]))
    topic = dspy.InputField(desc=fill_in_constants(input_descriptions_json["topic"]))
    existing_questions = dspy.InputField(desc=fill_in_constants(input_descriptions_json["existing_questions"]))
    sections = dspy.InputField(desc=fill_in_constants(input_descriptions_json["sections"]))
    additional_questions = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["additional_questions"]))
    # gaps = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["gaps"]))

# set the signature description
AddQuestionsToTopic.__doc__ = sig_description

print(AddQuestionsToTopic.__doc__)

# class FindGapsInTopic(dspy.Signature):

#     context = dspy.InputField(desc=fill_in_constants(input_descriptions_json["context"]))
#     topic = dspy.InputField(desc=fill_in_constants(input_descriptions_json["topic"]))
#     existing_questions = dspy.InputField(desc=fill_in_constants(input_descriptions_json["existing_questions"]))
#     gaps = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["gaps"]))

# # set the signature description
# FindGapsInTopic.__doc__ = sig_description

# print(FindGapsInTopic.__doc__)


You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided.
2. Please read through the existing questions for the inputted topic.
3. Identify any gaps that are related to the topic that the existing questions do not cover.
4. For each gap, generate two to three questions that would help the user elicit useful information from their constituents.
5. Generate time estimates in minutes for each question, which should populate the time_estimate field in the JSON output.
6. Please review the additional questions and compose a detailed explanation for how this question differs from the existing questions. Please keep your response between 20 and 50 words. Please start your response with "This question" followed by your rationale. For

In [9]:
# Create a module
class AddQuestionsToTopicModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.additional_questions = dspy.ChainOfThought(AddQuestionsToTopic)

        # self.gaps = dspy.ChainOfThought(FindGapsInTopic)

    def forward(self, context, topic, existing_questions, sections, return_rationale=False, temp=0.7):

        output = self.additional_questions(context=context, 
                                           topic=topic, 
                                           existing_questions=existing_questions,
                                             sections=sections,
                                           config=dict(temperature=temp))

        # output = self.gaps(context=context, 
        #                     topic=topic, 
        #                     existing_questions=existing_questions,
        #                     config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {
                    "additional_questions": output.additional_questions, 
                    # "gaps": output.gaps,
                    "rationale": output.rationale}
        else:
            return {
                    "additional_questions": output.additional_questions,
                    # "gaps": output.gaps
                    }

In [10]:
test_input_survey = """### What is the problem to be solved or the decision to be made?
The Parks Department ("PD") of a relatively small Massachusetts city (“Freeburg”) was recently granted state funds to make improvements to local parks. The PD doesn’t often receive grants of this size, so they want to make sure they use the funds effectively; if they use all the funds, they may be eligible for another grant next year.

Freeburg has 13 parks. Some are quite small, and would only require minimal improvements (e.g., tree-planting, de-weeding), whereas others will require major improvements to address safety and usability concerns.

Parks in wealthier neighborhoods of Freeburg tend to be nicer, which some residents believe may reflect a discrepancy in how tax funds are used and distributed by the city. The residents who take issue with these distributions tend to be lower-income and tend to live farther away from these parks, which have “higher-class” amenities, like tennis courts, public bathrooms, and water fountains with ground-level dog-bowl attachments. 

### What information is needed from the public to make the decision?
The parks were once well-kept, but in recent years, have been in a state of disarray, reflecting economic challenges that hit Freeburg hard during the COVID-19 pandemic. The PD needs to survey the residents of the Freeburg community to understand their needs, interests, and priorities as they relate to the local parks; this information will be used to inform what kinds of improvements are made to the parks.

The PD acknowledges that some improvements made by grant funds may lead to downstream costs that would not be covered by the grant, but the PD wishes to explore these anyway, due to their impact and long-term value for community members. For example, installing stationary trash and recycling bins in each park will help to reduce litter and improve the health and safety of the parks. However, while the state grant would pay for these bins to be installed, they would not pay for any future repairs or replacements, nor would they pay for the bins to be emptied regularly, which would be the task of the local Waste Management ("WM") service maintained by the city.

### What region is the engagement focused on? (e.g., city, county, state, national, etc.)
Freeburg

### Is the region urban, suburban, or rural? 
Suburban

### What groups of people will be affected by the outcome of the decision?
Some of the parks sit on the line with a nearby municipality, whose residents often use the parks. This may be viewed as either a challenge or opportunity by Freeburg residents, who may wish for the improved parks to be kept for their own private use, or who may wish for the parks to be shared (as they have been in the past) to expand the kinds of activities that the parks may host (for example, elementary school sporting events). 

There are several groups of constituents in the city, marked by demographic and geographic differences. Freeburg has a lower-altitude downtown (“DT”) that tends to have lower-income housing, in part due to historically long-standing social divisions, and in part due to the relatively high rate of flooding. The DT area has most of the city’s parks, but they tend to be far worse in quality, and are commonly policed (to many residents’ discomfort) to mitigate perceived issues with crime, which may or may not be the case. The DT area houses about 70% of the city’s residents, who are primarily from minority backgrounds. Freeburg also has a higher-altitude uptown (“UT”) area, whose residents tend to be higher-income. The UT area is the city’s financial and commerce district; as such, it brings in more out-of-city tourism and houses more of the city’s long-standing shopping (e.g., malls) entertainment venues (e.g., movie theaters). Residents of Freeburg are also divided by language. About 40% of the city’s residents are primarily Spanish-speaking, 8% are primarily Haitian-speaking, and 52% are primarily English-speaking. Throughout the city, signage (specifically, the languages used on public signage, such as those placed on parks) are an ongoing problem.

### Which of these groups are you engaging?
We will engage with residents in both the lower-altitude downtown (“DT”) and higher-altitude uptown (“UT”) areas.

### What form of engagement (e.g., virtual convenings, one-on-one interviews, focus groups, surveys) will best solicit the input needed from the communities you hope to engage?
Online survey

### What is the maximum amount of time in minutes you can expect people to spend on the engagement? (e.g., 5 minutes, 30 minutes, 60 minutes, etc.)
10 minutes maximum

### What is the breakdown of open-ended and close-ended questions?
20 percent of questions are open-ended and the remaining are close-ended"""

In [14]:
test_topic = "Favorite park activities"

test_existing_questions = [
    {
        "cell_type": "question",
        "response_format": "open",
        "description": "What is your favorite park activity?",
        "main_text": "What is your favorite park activity?",
        "response_categories": []
    },
    {
        "cell_type": "question",
        "response_format": "closed",
        "description": "Do you enjoy playing sports in the park?",
        "main_text": "Do you enjoy playing sports in the park?",
        "response_categories": [
            {"id": 0, "text": "Yes"},
            {"id": 1, "text": "No"}
        ]
    }
]

# convert test_existing_questions to a string
test_existing_questions_str = json.dumps(test_existing_questions)

test_sections = [
    {
        "id": 0,
        "title": "Introduction"
    },
    {
        "id": 1,
        "title": "Demographics"
    },
    {
        "id": 2,
        "title": "General Usage"
    },
    {
        "id": 3,
        "title": "Improvement Possibilities"
    }
]

# convert test_sections to a string
test_sections_str = json.dumps(test_sections)

In [15]:
# Test out AddQuestionsToTopicModule
test_module = AddQuestionsToTopicModule()

# Run the test input
output = test_module(context=test_input_survey, topic=test_topic, 
                     existing_questions=test_existing_questions_str, 
                        sections=test_sections_str,
                     return_rationale=True, temp=0.7001)

print(output)

{'additional_questions': '[\n  {\n    "section_id": 2,\n    "cell_type": "question",\n    "response_format": "closed",\n    "time_estimate": 1,\n    "rationale": "This question aims to understand the popularity of sports activities in the park.",\n    "main_text": "Which sports activities do you enjoy participating in at the park?",\n    "response_categories": [{"id": 0, "text": "Basketball"}, {"id": 1, "text": "Soccer"}, {"id": 2, "text": "Tennis"}, {"id": 3, "text": "Volleyball"}, {"id": 4, "text": "Other"}]\n  },\n  {\n    "section_id": 2,\n    "cell_type": "question",\n    "response_format": "closed",\n    "time_estimate": 2,\n    "rationale": "This question explores the variety of activities people are interested in beyond sports.",\n    "main_text": "Apart from sports, what other activities do you enjoy doing at the park?",\n    "response_categories": [{"id": 0, "text": "Picnicking"}, {"id": 1, "text": "Walking/Jogging"}, {"id": 2, "text": "Reading"}, {"id": 3, "text": "Socializi

In [17]:
gpt3_turbo.inspect_history(n=1)





You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided.
2. Please read through the existing questions for the inputted topic.
3. Identify any gaps that are related to the topic that the existing questions do not cover.
4. For each gap, generate two to three questions that would help the user elicit useful information from their constituents.
5. Generate time estimates in minutes for each question, which should populate the time_estimate field in the JSON output.
6. Please review the additional questions and compose a detailed explanation for how this question differs from the existing questions. Please keep your response between 20 and 50 words. Please start your response with "This question" followed by your rationale. 

### Signature for deleting questions for a topic

In [114]:
sig_description = """
You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided.
2. Please read through the existing questions for the inputted topic.
3. Identify an inputted number of questions in the existing set that could be removed. Consider the following factors when deciding which questions to remove, ordered by priority: (1) redundancy to other questions, (2) relevance to the topic, and (3) importance given the inputted context.
4. Generate time estimates in minutes for each question, which should populate the time_estimate field in the JSON output.
5. Please review the selected questions and compose a detailed explanation for why each question can be removed. If the rationale mentions the context, quote relevant parts of the context. If the rationale references another question, please include that question in the rationale instead of referencing the question number. Please keep your response between 20 and 50 words. Please start your response with "This question" followed by your rationale. For example: "This question can be deleted because..." These rationales should populate the rationale field in the JSON output.
"""

input_descriptions = """{
    "context": "The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question.",
    "topic": "The topic for which the user wants to generate questions.",
    "number_of_questions_to_remove": "The number of questions to remove from the existing set of questions.",
    "existing_questions": "The existing questions for the inputted topic. The questions are organized as a list of JSONs. Each JSON follows the following structure: {question_json_format}"
}
"""

output_descriptions = """{
    "questions_to_remove": "A subset of the existing questions that can be removed. The output should be a list of JSONs surrounded by square brackets, where each element has the following structure: {question_json_format}. Do not number the items in the list."
}"""

# output_descriptions = """{
#     "gaps": "A list of gaps in the existing questions. The list should be a string where each gap is separated by a semicolon. Do not number the items in the list."
# }"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class RemoveQuestionsInTopic(dspy.Signature):

    context = dspy.InputField(desc=fill_in_constants(input_descriptions_json["context"]))
    topic = dspy.InputField(desc=fill_in_constants(input_descriptions_json["topic"]))
    number_of_questions_to_remove = dspy.InputField(desc=fill_in_constants(input_descriptions_json["number_of_questions_to_remove"]))
    existing_questions = dspy.InputField(desc=fill_in_constants(input_descriptions_json["existing_questions"]))
    questions_to_remove = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["questions_to_remove"]))

# set the signature description
RemoveQuestionsInTopic.__doc__ = sig_description

print(RemoveQuestionsInTopic.__doc__)



You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided.
2. Please read through the existing questions for the inputted topic.
3. Identify an inputted number of questions in the existing set that could be removed. Consider the following factors when deciding which questions to remove, ordered by priority: (1) redundancy to other questions, (2) relevance to the topic, and (3) importance given the inputted context.
4. Generate time estimates in minutes for each question, which should populate the time_estimate field in the JSON output.
5. Please review the selected questions and compose a detailed explanation for why each question can be removed. If the rationale mentions the context, quote relevant parts of the context. If th

In [115]:
# Create a module
class RemoveQuestionsInTopicModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.questions_to_remove = dspy.ChainOfThought(RemoveQuestionsInTopic)

    def forward(self, context, topic, number_of_questions_to_remove, existing_questions, return_rationale=False, temp=0.7):

        output = self.questions_to_remove(context=context, 
                                           topic=topic, 
                                           number_of_questions_to_remove=number_of_questions_to_remove,
                                           existing_questions=existing_questions,
                                           config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {
                    "questions_to_remove": output.questions_to_remove, 
                    "rationale": output.rationale}
        else:
            return {
                    "questions_to_remove": output.questions_to_remove,
                    }

In [116]:
test_input_survey = """### What is the problem to be solved or the decision to be made?
The Parks Department ("PD") of a relatively small Massachusetts city (“Freeburg”) was recently granted state funds to make improvements to local parks. The PD doesn’t often receive grants of this size, so they want to make sure they use the funds effectively; if they use all the funds, they may be eligible for another grant next year.

Freeburg has 13 parks. Some are quite small, and would only require minimal improvements (e.g., tree-planting, de-weeding), whereas others will require major improvements to address safety and usability concerns.

Parks in wealthier neighborhoods of Freeburg tend to be nicer, which some residents believe may reflect a discrepancy in how tax funds are used and distributed by the city. The residents who take issue with these distributions tend to be lower-income and tend to live farther away from these parks, which have “higher-class” amenities, like tennis courts, public bathrooms, and water fountains with ground-level dog-bowl attachments. 

### What information is needed from the public to make the decision?
The parks were once well-kept, but in recent years, have been in a state of disarray, reflecting economic challenges that hit Freeburg hard during the COVID-19 pandemic. The PD needs to survey the residents of the Freeburg community to understand their needs, interests, and priorities as they relate to the local parks; this information will be used to inform what kinds of improvements are made to the parks.

The PD acknowledges that some improvements made by grant funds may lead to downstream costs that would not be covered by the grant, but the PD wishes to explore these anyway, due to their impact and long-term value for community members. For example, installing stationary trash and recycling bins in each park will help to reduce litter and improve the health and safety of the parks. However, while the state grant would pay for these bins to be installed, they would not pay for any future repairs or replacements, nor would they pay for the bins to be emptied regularly, which would be the task of the local Waste Management ("WM") service maintained by the city.

### What region is the engagement focused on? (e.g., city, county, state, national, etc.)
Freeburg

### Is the region urban, suburban, or rural? 
Suburban

### What groups of people will be affected by the outcome of the decision?
Some of the parks sit on the line with a nearby municipality, whose residents often use the parks. This may be viewed as either a challenge or opportunity by Freeburg residents, who may wish for the improved parks to be kept for their own private use, or who may wish for the parks to be shared (as they have been in the past) to expand the kinds of activities that the parks may host (for example, elementary school sporting events). 

There are several groups of constituents in the city, marked by demographic and geographic differences. Freeburg has a lower-altitude downtown (“DT”) that tends to have lower-income housing, in part due to historically long-standing social divisions, and in part due to the relatively high rate of flooding. The DT area has most of the city’s parks, but they tend to be far worse in quality, and are commonly policed (to many residents’ discomfort) to mitigate perceived issues with crime, which may or may not be the case. The DT area houses about 70% of the city’s residents, who are primarily from minority backgrounds. Freeburg also has a higher-altitude uptown (“UT”) area, whose residents tend to be higher-income. The UT area is the city’s financial and commerce district; as such, it brings in more out-of-city tourism and houses more of the city’s long-standing shopping (e.g., malls) entertainment venues (e.g., movie theaters). Residents of Freeburg are also divided by language. About 40% of the city’s residents are primarily Spanish-speaking, 8% are primarily Haitian-speaking, and 52% are primarily English-speaking. Throughout the city, signage (specifically, the languages used on public signage, such as those placed on parks) are an ongoing problem.

### Which of these groups are you engaging?
We will engage with residents in both the lower-altitude downtown (“DT”) and higher-altitude uptown (“UT”) areas.

### What form of engagement (e.g., virtual convenings, one-on-one interviews, focus groups, surveys) will best solicit the input needed from the communities you hope to engage?
Online survey

### What is the maximum amount of time in minutes you can expect people to spend on the engagement? (e.g., 5 minutes, 30 minutes, 60 minutes, etc.)
10 minutes maximum

### What is the breakdown of open-ended and close-ended questions?
20 percent of questions are open-ended and the remaining are close-ended"""

In [118]:
test_topic = "Favorite park activities"

test_existing_questions = [
    {
        "cell_type": "question",
        "response_format": "open",
        "description": "",
        "main_text": "What is your favorite park activity?",
        "response_categories": []
    },
    {
        "cell_type": "question",
        "response_format": "closed",
        "description": "",
        "main_text": "Do you enjoy playing sports in the park?",
        "response_categories": [
            {"id": 0, "text": "Yes"},
            {"id": 1, "text": "No"}
        ]
    },
    {
        "cell_type": "question",
        "response_format": "closed",
        "description": "",
        "main_text": "Do you enjoy reading in the park?",
        "response_categories": [
            {"id": 0, "text": "Yes"},
            {"id": 1, "text": "No"}
        ]
    },
    {
        "cell_type": "question",
        "response_format": "closed",
        "description": "",
        "main_text": "Do you enjoy walking in the park?",
        "response_categories": [
            {"id": 0, "text": "Yes"},
            {"id": 1, "text": "No"}
        ]
    },
    {
        "cell_type": "question",
        "response_format": "closed",
        "description": "",
        "main_text": "Do you enjoy biking in the park?",
        "response_categories": [
            {"id": 0, "text": "Yes"},
            {"id": 1, "text": "No"}
        ]
    },
    {
        "cell_type": "question",
        "response_format": "open",
        "description": "",
        "main_text": "What is your favorite thing to do in the park?",
        "response_categories": []
    }
]

# convert test_existing_questions to a string
test_existing_questions_str = json.dumps(test_existing_questions)

In [121]:
# Test out RemoveQuestionsInTopicModule
test_module = RemoveQuestionsInTopicModule()

# Run the test input
output = test_module(context=test_input_survey, topic=test_topic, 
                     number_of_questions_to_remove="3",
                     existing_questions=test_existing_questions_str, 
                     return_rationale=True, temp=0.7001)

print(output)

{'questions_to_remove': '[\n    {\n        "cell_type": "question",\n        "response_format": "closed",\n        "time_estimate": 1,\n        "rationale": "This question can be deleted as it overlaps with the question \'Do you enjoy playing sports in the park?\', and the context indicates more relevant aspects to focus on.",\n        "main_text": "Do you enjoy reading in the park?",\n        "response_categories": [\n            {"id": 0, "text": "Yes"}, \n            {"id": 1, "text": "No"}\n        ]\n    },\n    {\n        "cell_type": "question",\n        "response_format": "closed",\n        "time_estimate": 1,\n        "rationale": "This question can be removed because walking in the park is a common activity that may not provide distinct insights compared to other questions.",\n        "main_text": "Do you enjoy walking in the park?",\n        "response_categories": [\n            {"id": 0, "text": "Yes"}, \n            {"id": 1, "text": "No"}\n        ]\n    },\n    {\n      

In [122]:
gpt3_turbo.inspect_history(n=1)





You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided.
2. Please read through the existing questions for the inputted topic.
3. Identify an inputted number of questions in the existing set that could be removed. Consider the following factors when deciding which questions to remove, ordered by priority: (1) redundancy to other questions, (2) relevance to the topic, and (3) importance given the inputted context.
4. Generate time estimates in minutes for each question, which should populate the time_estimate field in the JSON output.
5. Please review the selected questions and compose a detailed explanation for why each question can be removed. If the rationale mentions the context, quote relevant parts of the context. If